# Nạp chunk vào IVFPQ index đã train

Input: `index.faiss` từ notebook `FAISS_IVFPQ_Index.ipynb` (đã train, chưa thêm vector) và `merged.json`: mảng tài liệu, mỗi tài liệu có `doc_id`, `status`, `url`, `parents` và `children`. Mỗi child là một chunk có `chunk_text`, `start_char`, `end_char` và các ID liên quan.

Output trong `/kaggle/working`: **`index.faiss`** sau khi add và **`metadata.json`**.

Bật GPU trong Kaggle Settings, sửa đường dẫn rồi chạy các cell từ trên xuống. Cần `torch`, `sentence-transformers`, `numpy`, `faiss`; notebook không tự cài thư viện. Bản `faiss-cpu` vẫn dùng được GPU để embedding, nhưng bước add chạy CPU.

## 1. Cấu hình

`start_point`, `end_point` là khoảng `doc_id`, gồm cả hai đầu. `end_point = -1` nghĩa là không giới hạn trên.

File `merged.json` hiện tại có 3.472.347 tài liệu và 19.157.144 child chunk. Notebook đọc định dạng đã xuất: dấu `[`/`]` ở dòng riêng, mỗi tài liệu JSON nằm trên một dòng. Không dùng `json.load()` cho toàn bộ file khoảng 24,6 GB.

In [ ]:
import json
from pathlib import Path
from typing import Iterator, TextIO

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

index_path = Path("/kaggle/input/YOUR_DATASET/index.faiss")
data_path = Path("/kaggle/input/YOUR_DATASET/merged.json")
output_dir = Path("/kaggle/working")

start_point = 0
end_point = -1
embedding_model = "Alibaba-NLP/gte-multilingual-base"
batch_size = 128
add_batch_size = 8_192
use_fp16 = True
use_faiss_gpu = True

output_index = output_dir / "index.faiss"
output_metadata = output_dir / "metadata.json"
output_dir.mkdir(parents=True, exist_ok=True)
if output_index.exists() or output_metadata.exists():
    raise FileExistsError("Choose another output_dir to preserve existing output files")

## 2. Đọc index và tải model

Giữ cùng model và chuẩn hóa L2 như notebook train. `use_fp16=True` dùng mixed precision khi embedding trên GPU để giảm VRAM và tăng tốc; đặt `False` để embedding bằng FP32 như lúc train.

FAISS GPU dùng bảng PQ FP16 vì index `M=64, nbits=8` cần thiết lập này theo [tài liệu FAISS GPU](https://github.com/facebookresearch/faiss/wiki/Faiss-on-the-GPU). Nếu FAISS GPU không có hoặc không chuyển được index, notebook báo lý do và dùng FAISS CPU. Mỗi lần chạy dùng GPU số 0.

In [ ]:
index = faiss.read_index(str(index_path))
if not index.is_trained or index.ntotal != 0:
    raise ValueError("Use the trained, empty index from FAISS_IVFPQ_Index.ipynb")
index.nprobe = 16

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Embedding device: {device}")
if device.type == "cpu":
    print("CUDA is unavailable; embedding runs on CPU. Enable GPU in Kaggle Settings.")
model = SentenceTransformer(embedding_model, trust_remote_code=True, device=str(device))
model.eval()

faiss_on_gpu = False
if use_faiss_gpu and device.type == "cuda" and hasattr(faiss, "StandardGpuResources"):
    try:
        gpu_resources = faiss.StandardGpuResources()
        gpu_resources.setTempMemory(256 * 1024**2)
        gpu_options = faiss.GpuClonerOptions()
        gpu_options.useFloat16 = True  # Required for 64-byte PQ codes on common Kaggle GPUs.
        index = faiss.index_cpu_to_gpu(gpu_resources, 0, index, gpu_options)
        faiss_on_gpu = True
    except RuntimeError as error:
        print(f"FAISS GPU conversion failed; using CPU: {error}")
print(f"FAISS device: {'GPU 0' if faiss_on_gpu else 'CPU'}")
if use_faiss_gpu and not faiss_on_gpu:
    print("FAISS add uses CPU because CUDA or a compatible FAISS GPU build is unavailable.")

## 3. Embedding, add vector và ghi metadata

Đọc lần lượt từng tài liệu trong `merged.json`, lọc theo `doc_id` của tài liệu, rồi lấy từng chunk trong `children`. Bỏ tài liệu không có `status="ok"`, chunk không đủ điều kiện retrieval và nội dung rỗng. Không gộp các child thành một văn bản. Metadata được ghi từng batch nên RAM phụ thuộc kích thước một tài liệu và batch, thay vì toàn bộ dữ liệu.

ID vector là số nguyên liên tiếp từ 0. Các chunk cùng `doc_id` vẫn có ID riêng. Giữ `chunk_id`, `parent_id`, `url` để truy ngược nguồn; `start_char`/`end_char` được đổi tên thành `char_start`/`char_end` để giữ cấu trúc metadata cũ. `metadata.json` có dạng:

```json
{"0": {"doc_id": 101003, "chunk_order": 0, "char_start": 0, "char_end": 844, "chunk_text": "Nội dung chunk...", "chunk_id": "101003:c:0:0:844:paragraph-v3", "parent_id": "101003:p:0:0:4111:paragraph-v3", "url": "https://example.com/article"}}
```

Khi search trả về ID `i`, lấy record bằng `metadata[str(i)]`. Nếu GPU hết VRAM, giảm `batch_size` xuống 64 hoặc 32; `add_batch_size` là số record gom mỗi lượt, còn `batch_size` là số text model embedding cùng lúc.


In [ ]:
def iter_merged_documents(data_file: TextIO) -> Iterator[dict]:
    """Stream the exported array: one document per line, with separate brackets."""
    if data_file.readline().strip() != "[":
        raise ValueError("Expected merged.json with '[' on its own first line")
    documents_seen = False
    previous_comma = True
    for line_number, line in enumerate(data_file, start=2):
        raw = line.strip()
        if not raw:
            continue
        if raw == "]":
            if documents_seen and previous_comma:
                raise ValueError("Trailing comma in merged.json")
            if any(line.strip() for line in data_file):
                raise ValueError("Unexpected content after merged.json array")
            return
        if not previous_comma:
            raise ValueError(f"Missing document separator at line {line_number}")
        previous_comma = raw.endswith(",")
        if previous_comma:
            raw = raw[:-1]
        try:
            document = json.loads(raw)
        except json.JSONDecodeError as error:
            raise ValueError(f"Invalid document JSON at line {line_number}") from error
        if not isinstance(document, dict):
            raise ValueError(f"Expected a document object at line {line_number}")
        documents_seen = True
        yield document
    raise ValueError("Missing closing ']' in merged.json")


def iter_chunk_records(
    data_file: TextIO, start_doc_id: int = 0, end_doc_id: int = -1,
) -> Iterator[dict]:
    """Flatten eligible children and preserve the existing metadata field names."""
    if end_doc_id != -1 and end_doc_id < start_doc_id:
        raise ValueError("end_doc_id must be -1 or at least start_doc_id")
    for document in iter_merged_documents(data_file):
        doc_id = document["doc_id"]
        if doc_id < start_doc_id or (end_doc_id != -1 and doc_id > end_doc_id):
            continue
        if document.get("status") != "ok":
            continue
        for chunk in document["children"]:
            if not chunk.get("retrieval_eligible", True):
                continue
            text = chunk["chunk_text"]
            if not isinstance(text, str):
                raise ValueError(f"Non-string chunk_text in document {doc_id}")
            if not text.strip():
                continue
            if chunk["doc_id"] != doc_id:
                raise ValueError(f"Child doc_id differs from document {doc_id}")
            yield {
                "doc_id": doc_id,
                "chunk_order": chunk["chunk_order"],
                "char_start": chunk["start_char"],
                "char_end": chunk["end_char"],
                "chunk_text": text,
                "chunk_id": chunk["chunk_id"],
                "parent_id": chunk["parent_id"],
                "url": document["url"],
            }


def add_batch(records: list[dict], metadata_file: TextIO) -> None:
    """Embed one batch, add unique vector IDs, and write matching metadata."""
    texts = [record["chunk_text"] for record in records]
    with torch.inference_mode(), torch.autocast(
        device_type=device.type, dtype=torch.float16,
        enabled=use_fp16 and device.type == "cuda",
    ):
        vectors = model.encode(
            texts, batch_size=batch_size, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=False,
        )
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    faiss.normalize_L2(vectors)
    first_id = index.ntotal
    ids = np.arange(first_id, first_id + len(records), dtype=np.int64)
    index.add_with_ids(vectors, ids)

    if first_id:
        metadata_file.write(",")
    metadata_file.write(",".join(
        f'"{int(vector_id)}":' + json.dumps(record, ensure_ascii=False, separators=(",", ":"))
        for vector_id, record in zip(ids, records)
    ))
    print(f"\rVectors added: {index.ntotal:,}", end="", flush=True)


batch = []
with data_path.open(encoding="utf-8") as data_file, output_metadata.open("x", encoding="utf-8") as metadata_file:
    metadata_file.write("{")
    for record in iter_chunk_records(data_file, start_point, end_point):
        batch.append(record)
        if len(batch) == add_batch_size:
            add_batch(batch, metadata_file)
            batch.clear()
    if batch:
        add_batch(batch, metadata_file)
        batch.clear()
    metadata_file.write("}")
print(f"\nTotal vectors: {index.ntotal:,}")

## 4. Lưu index sau khi add

Chuyển index GPU về CPU để ghi file `.faiss`. File index đầu vào được giữ nguyên; hai output nằm trong `output_dir`. Nếu chạy lại toàn bộ, chọn thư mục output mới để giữ kết quả cũ.

In [ ]:
if output_index.exists():
    raise FileExistsError(f"Output already exists: {output_index}")
index_to_save = faiss.index_gpu_to_cpu(index) if faiss_on_gpu else index
faiss.write_index(index_to_save, str(output_index))
print(f"Index: {output_index}")
print(f"Metadata: {output_metadata}")